# Week4_ex2 - Induction Heating (Ferrite)

Same induction-heating setup as ex1 but with a ferrite core instead of stainless steel - originally used the Transient solver but that's not supported on the Student license, so switched to EddyCurrent since there's no moving parts and the current is a single fixed frequency anyway. Ohmic loss and B-field now come out as steady-state values at 10Hz instead of a time waveform.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
# 1. AEDT desktop interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create the project and set the solution type - EddyCurrent instead of Transient, Student license doesn't support Transient at all
sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object, needed to reuse script-recording snippets
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week4_ex2"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

desi_name = "Week4_ex2"
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()


In [ ]:
core_thickness = 40
M3D["core_thickness"] = f"{core_thickness}mm"

len_x = core_thickness
M3D["len_x"] = f"{len_x}mm"

len_y = 200
M3D["len_y"] = f"{len_y}mm"

len_z = 150
M3D["len_z"] = f"{len_z}mm"

coil_len = len_x + 6
M3D["coil_len"] = f"{coil_len}mm"

coil_gap = 5
M3D["coil_gap"] = f"{coil_gap}mm"

coil_num = 5
M3D["coil_num"] = f"{coil_num}"

frequency1 = 10
M3D["frequency1"] = f"{frequency1}"

# EddyCurrent takes the current as a magnitude (phasor), not a time-domain expression -
# the frequency itself now lives in the setup instead of inside the current formula
M3D["current"] = "100A"  # peak magnitude, phase is set separately in assign_winding

In [ ]:
# Setup created via script-recording style call (bypasses pyaedt's create_setup() /
# M3D.setups[-1], which raise KeyError: 'ac magnetic' on AEDT 2025 R2 due to the
# EddyCurrent -> AC Magnetic rename)
setup_name = "Setup1"
oModule = oDesign.GetModule("AnalysisSetup")

oModule.InsertSetup("EddyCurrent", 
	[
		"NAME:Setup1",
		"Enabled:=", True,
		[
			"NAME:MeshLink",
			"ImportMesh:=", False
		],
		"MaximumPasses:=", 15,
		"MinimumPasses:=", 2,
		"MinimumConvergedPasses:=", 1,
		"PercentRefinement:=", 30,
		"SolveFieldOnly:=", False,
		"PercentError:=", 1,
		"SolveMatrixAtLast:=", True,
		"UseNonLinearIterNum:=", False,
		"CacheSaveKind:=", "Delta",
		"ConstantDelta:=", "0s",
		"UseCacheFor:=", ["Freq"],
		"UseIterativeSolver:=", False,
		"RelativeResidual:=", 1E-05,
		"NonLinearResidual:=", 0.0001,
		"RelaxationFactor:=", 1,
		"SmoothBHCurve:=", True,
		"Frequency:=", "10Hz",
		"HasSweepSetup:=", False,
		"UseHighOrderShapeFunc:=", False,
		"ImportMeshForMuLink:=", False,
		"LossAdaptiveCtrl:=", "0.5",
		"UseMuLink:=", False
	])


In [ ]:
core_ferrite = M3D.materials.duplicate_material("Ferrite", name="core_ferrite")  # material for the core

coil_copper = M3D.materials.duplicate_material("copper", name="coil_copper")  # material for the coil

In [ ]:
## geometry and boundary setup ##

#### 1. draw the core
box_origin = ["-len_x/2", "-len_y/2", "-len_z/2"]
box_sizes = ["len_x", "len_y", "len_z"]
core = M3D.modeler.create_box(origin=box_origin, sizes=box_sizes, name="Core", material="core_ferrite")

box_origin = ["-len_x/2", "-(len_y-2*core_thickness)/2", "-(len_z-2*core_thickness)/2"]
box_sizes = ["len_x", "(len_y-2*core_thickness)", "(len_z-2*core_thickness)"]
core_hole = M3D.modeler.create_box(origin=box_origin, sizes=box_sizes, name="core_hole", material=None)

center = ["0mm", "-len_y/2+20mm", "-15mm"]
origin = ["40mm", "-len_y/2+20mm", "-15mm"]
core_hole2 = M3D.modeler.create_polyhedron(orientation=None, center=center, origin=origin, height="30mm", num_sides=36, name="core_hole2", material=None)

center = ["0mm", "-len_y/2+20mm", "-15mm"]
origin = ["20mm", "-len_y/2+20mm", "-15mm"]
core_cal = M3D.modeler.create_polyhedron(orientation=None, center=center, origin=origin, height="30mm", num_sides=36, name="Core_cal", material="core_ferrite")

core.subtract(tool_list=[core_hole], keep_originals=False)
core.subtract(tool_list=[core_hole2], keep_originals=False)


# 2. draw the straight wire
points = []
points.append(["coil_len/2", "-(coil_len/2+100mm)", 0])
for n in range(coil_num) :
    points.append(["coil_len/2", "-coil_len/2", f"{n}*coil_gap"])
    points.append(["coil_len/2", "coil_len/2", f"{n}*coil_gap"])
    points.append(["-coil_len/2", "coil_len/2", f"{n}*coil_gap"])
    points.append(["-coil_len/2", "-coil_len/2", f"{n}*coil_gap"])
points.append(["-coil_len/2", "-(coil_len/2+100mm)", "(coil_num-1)*coil_gap"])

coil = M3D.modeler.create_polyline(points=points, name="Coil", material="coil_copper", 
                                   xsection_type="Circle", xsection_width="2mm", xsection_num_seg=6)


vector = ["0mm", "-(len_y-core_thickness)/2", "-( coil_gap*(coil_num-1) )/2" ]
coil.move(vector)

# 3. create the region
region = M3D.modeler.create_region(pad_value=["400", "400", "50", "0", "200", "200"], pad_type='Percentage Offset', name='Region')


# 4. eddy current setting - only the core needs this, the coil should stay off
M3D.eddy_effects_on(assignment=core.name, enable_eddy_effects=True, enable_displacement_current=True)

M3D.eddy_effects_on(assignment=core_cal.name, enable_eddy_effects=True, enable_displacement_current=True)
    
M3D.eddy_effects_on(assignment=coil.name, enable_eddy_effects=False, enable_displacement_current=False)

In [ ]:
# static solvers (Magnetostatic, EddyCurrent) auto-generate adaptive mesh from the setup, so
# the manual AssignLengthOp from the Transient version isn't needed here. Per mesh policy,
# only add mesh sizing back if a cap error actually shows up during local execution
# (known range so far: ~32,330 elements passed, ~64,484+ failed)

In [ ]:
# 1. find the terminal faces on the coil
coil_terminal = [0, 0]
for f in coil.faces :   # picking the terminal faces by their center coordinates
    if (f.center[1]<(-0.75*len_y))&(f.center[2]>0) :
        coil_terminal[0] = f
    
    if (f.center[1]<(-0.75*len_y))&(f.center[2]<0) :
        coil_terminal[1] = f

# assign_coil / assign_winding / add_winding_coils haven't been verified locally yet against
# the AC Magnetic rename bug for this solver - keep an eye on this cell first if it errors
coil_1 = M3D.assign_coil(assignment=coil_terminal[0], conductors_number=1, polarity='Positive', name=None)
coil_2 = M3D.assign_coil(assignment=coil_terminal[1], conductors_number=1, polarity='Negative', name=None)


# 2. create a winding and add both coil terminals to it - current is a magnitude + phase now
# instead of a time expression, since EddyCurrent solves one fixed frequency in steady state
current1 = M3D.assign_winding(assignment=None, winding_type='Current', is_solid=True, current="current", parallel_branches=1, phase=0, name=None)
M3D.add_winding_coils(assignment=current1.name, coils=[coil_1.name, coil_2.name])

In [ ]:
oModule = oDesign.GetModule("FieldsReporter")

oModule.AddNamedExpression("core_ohmic_loss", "Fields", 
	[
		"NameOfExpression:="	, ["Ohmic_Loss"],
		"EnterVolume:="		, [core_cal.name],
		"Operation:="		, ["VolumeValue"],
		"Operation:="		, ["Integrate"]
	])  # ohmic loss inside the calculation volume

oModule.AddNamedExpression("core_b_field", "Fields", 
	[
		"NameOfExpression:="	, ["Mag_B"],
		"EnterVolume:="		, [core_cal.name],
		"Operation:="		, ["VolumeValue"],
		"Operation:="		, ["Mean"]
	])  # average B-field inside the calculation volume

In [ ]:
oDesign.Analyze(setup_name)


In [ ]:
# NOTE: get_solution_data() (pyaedt wrapper) internally looks up the report object by
# solution type, and fails the same way create_setup()/analyze()/assign_radiation() do
# elsewhere in this project -- AEDT 2025 R2 reports this solver as "AC Magnetic" internally,
# pyaedt 0.15.3 doesn't recognize that string, and get_solution_data() silently returns
# False instead of raising, which then breaks on .data_real(). Same root cause as the other
# raw-API workarounds in this notebook, just surfacing in a new pyaedt call this time.
#
# Bypassing it here the same way: push each Named Expression onto the Field Calculator
# stack via the raw API and read the value back directly, exactly like doing it by hand
# in Fields Calculator (Copy to stack -> Eval).
oModule = oDesign.GetModule("FieldsReporter")

oModule.CopyNamedExprToStack("core_ohmic_loss")
core_ohmic_loss_val = oModule.GetTopEntryValue("Setup1 : LastAdaptive", ["Freq:=", "10Hz", "Phase:=", "0deg"])[0]
oModule.CalcStack("clear")

oModule.CopyNamedExprToStack("core_b_field")
core_b_field_val = oModule.GetTopEntryValue("Setup1 : LastAdaptive", ["Freq:=", "10Hz", "Phase:=", "0deg"])[0]
oModule.CalcStack("clear")

result = {
    "core_thickness_mm": core_thickness,
    "core_ohmic_loss": core_ohmic_loss_val,
    "core_b_field": core_b_field_val,
}
print(result)

df = pd.DataFrame([result])
csv_path = "Week4_ex2_result.csv"  # saved flat, next to the notebook -- same convention as Images/
df.to_csv(csv_path, index=False)
df

In [ ]:
M3D.save_project()